 Live survey response analysis

In [9]:
import json
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

from survey_responce import (
	extract_triplets_by_category,
	limit_response_cases,
	rank_borda,
	rank_by_category,
	rank_overall,
	rank_plackett_luce,
	summarize_participants,
)


In [10]:
config_path = "live-analisys-config.json"
survey_path = "../eval/output/survey.json"

with open(config_path, encoding="utf-8") as config_file:
	config = json.load(config_file)


def fetch_api(path):
	request = Request(
		f"{config['base_url'].rstrip('/')}{path}",
		headers={"Authorization": f"Bearer {config['admin_token']}"},
	)
	with urlopen(request, timeout=30) as api_response:
		return json.load(api_response)


n_cases = 99
responses = fetch_api("/api/responses")
response_labels = pd.Series([response.get("label") for response in responses], dtype="string")
responses = [
	response
	for response, include in zip(responses, response_labels.str.match(r"^p\d{2}", na=False))
	if include
]
responses = limit_response_cases(responses, survey_path, n_cases)
participants = fetch_api("/api/participants/")


 Responses by participant

In [11]:
participant_stats = pd.DataFrame(
	summarize_participants(responses, participants, survey_path),
	columns=[
		"participant", "group", "status", "last_page", "saved_at",
		"completed_cases", "incomplete_cases", "complete_rankings",
	],
)
participant_stats["group"] = pd.to_numeric(participant_stats["group"]).astype("Int64")
participant_stats = participant_stats.astype({"last_page": "Int64"})
status_counts = participant_stats["status"].value_counts()
display(Markdown(
	f"**{len(participant_stats)}** participants with responses: "
	f"**{status_counts.get('submitted', 0)}** submitted, **{status_counts.get('draft', 0)}** drafts; "
	f"**{participant_stats['completed_cases'].sum()}** completed cases and "
	f"**{participant_stats['complete_rankings'].sum()}** complete rankings in total."
))
participant_stats.sort_values(["status", "participant"], ascending=[False, True], ignore_index=True)


**7** participants with responses: **0** submitted, **7** drafts; **42** completed cases and **255** complete rankings in total.

,participant,group,status,last_page,saved_at,completed_cases,incomplete_cases,complete_rankings
0,p01,1,draft,1,2026-10-04T17:59:30.117000,1,0,6
1,p02,2,draft,0,2026-10-04T18:03:15.580000,1,0,6
2,p04,4,draft,1,2026-10-04T17:59:04.500000,1,0,6
3,p05,5,draft,1,2026-10-04T18:23:31.607000,1,0,6
4,p07,7,draft,28,2026-10-06T10:09:35.867000,28,0,168
5,p08,8,draft,4,2026-10-05T10:12:24.710000,4,0,24
6,p09,9,draft,6,2026-10-06T10:17:58.273000,6,1,39


Rankings by category

In [12]:
# Drafts are autosaved on every change, so rows without both best and worst are skipped.
triplets_by_category = extract_triplets_by_category(responses, survey_path, skip_incomplete=True)
rank_method = rank_borda
# rank_method = rank_plackett_luce 

for category, rankings in rank_by_category(triplets_by_category, rank_method).items():
	display(Markdown(f"### {category}"))
	display(pd.DataFrame(rankings).set_index("rank"))

if triplets_by_category:
	display(Markdown("### Overall"))
	display(pd.DataFrame(rank_overall(triplets_by_category, rank_method)).set_index("rank"))
else:
	display(Markdown("No complete rankings yet."))


### Tačnost navedenih informacija

,model,score,answer_count
rank,,,
1,gpt-5.2,3.846154,26
2,Qwen/Qwen3-32B,3.416667,24
3,Qwen/Qwen3-14B-ft,2.777778,27
4,Qwen/Qwen3-14B,2.760000,25
5,nvidia/Llama-3.3-70B-Instruct-FP8,2.259259,27


### Usklađenost sa gradivom

,model,score,answer_count
rank,,,
1,gpt-5.2,3.615385,26
2,Qwen/Qwen3-32B,3.333333,24
3,Qwen/Qwen3-14B,3.080000,25
4,Qwen/Qwen3-14B-ft,2.925926,27
5,nvidia/Llama-3.3-70B-Instruct-FP8,2.111111,27


### Primerenost nivou znanja i mogućnostima učenika

...

,model,score,answer_count
rank,,,
1,gpt-5.2,3.640000,25
2,Qwen/Qwen3-14B-ft,3.384615,26
3,nvidia/Llama-3.3-70B-Instruct-FP8,3.074074,27
4,Qwen/Qwen3-14B,2.520000,25
5,Qwen/Qwen3-32B,2.304348,23


### Koji biste odgovor najradije iskoristili u praksi, uz moguće dorade?

,model,score,answer_count
rank,,,
1,gpt-5.2,3.800000,25
2,Qwen/Qwen3-32B,3.260870,23
3,Qwen/Qwen3-14B-ft,3.000000,26
4,Qwen/Qwen3-14B,2.760000,25
5,nvidia/Llama-3.3-70B-Instruct-FP8,2.259259,27


### Overall

,model,score,answer_count
rank,,,
1,gpt-5.2,3.705882,153
2,Qwen/Qwen3-32B,3.085106,141
3,Qwen/Qwen3-14B-ft,3.050314,159
4,Qwen/Qwen3-14B,2.800000,150
5,nvidia/Llama-3.3-70B-Instruct-FP8,2.395062,162
